See our docs for an explanation of what this code is doing!

In [ ]:
import os
from docent import Docent

client = Docent(
    email=os.getenv("DOCENT_EMAIL"),        # is default and can be omitted
    password=os.getenv("DOCENT_PASSWORD"),  # is default and can be omitted

    # Uncomment and adjust these if you're self-hosting
    # server_url="http://localhost:8889",
    # web_url="http://localhost:3001",
)

In [ ]:
fg_id = client.create_framegrid(name="inspect example", description="example inspect log that comes with the Docent repo")

In [ ]:
from docent.samples import get_inspect_fpath
from inspect_ai.log import read_eval_log
from pydantic_core import to_jsonable_python

ctf_log = read_eval_log(get_inspect_fpath())
ctf_log_dict = to_jsonable_python(ctf_log)

In [ ]:
from typing import Any
from docent.data_models import BaseAgentRunMetadata
from pydantic import Field

class InspectAgentRunMetadata(BaseAgentRunMetadata):
    task_id: str = Field(
        description="The ID of the 'benchmark' or 'set of evals' that the transcript belongs to"
    )

    # Identification of this particular run
    sample_id: str = Field(
        description="The specific task inside of the `task_id` benchmark that the transcript was run on"
    )
    epoch_id: int = Field(
        description="Each `sample_id` should be run multiple times due to stochasticity; `epoch_id` is the integer index of a specific run."
    )

    # Parameters for the run
    model: str = Field(description="The model that was used to generate the transcript")

    # Outcome
    scoring_metadata: dict[str, Any] | None = Field(
        description="Additional metadata about the scoring process"
    )

    # Inspect metadata
    additional_metadata: dict[str, Any] | None = Field(
        description="Additional metadata about the transcript"
    )

In [ ]:
from inspect_ai.log import EvalLog
from docent.data_models import AgentRun, Transcript
from docent.data_models.chat import parse_chat_message

def load_inspect_log(log: EvalLog) -> list[AgentRun]:
    if log.samples is None:
        return []

    agent_runs: list[AgentRun] = []

    for s in log.samples:
        # Extract sample_id from the sample ID
        sample_id = s.id
        epoch_id = s.epoch

        # Gather scores
        scores: dict[str, int | float | bool | None] = {}

        # Evaluate correctness (for this CTF benchmark)
        if s.scores and "includes" in s.scores:
            scores["correct"] = s.scores["includes"].value == "C"

        # Set metadata
        metadata = InspectAgentRunMetadata(
            task_id=log.eval.task,
            sample_id=str(sample_id),
            epoch_id=epoch_id,
            model=log.eval.model,
            scores=scores,
            additional_metadata=s.metadata,
            scoring_metadata=s.scores,
        )

        # Create transcript
        agent_runs.append(
            AgentRun(
                transcripts={
                    "default": Transcript(
                        messages=[parse_chat_message(m.model_dump()) for m in s.messages]
                    )
                },
                metadata=metadata,
            )
        )

    return agent_runs

In [ ]:
agent_runs = load_inspect_log(ctf_log)
print(agent_runs[0].text)

In [ ]:
client.add_agent_runs(fg_id, agent_runs)